# 05 — gammapy fit recovery

Closed-loop validation of kernel -> model -> gammapy fitting together: simulate a
`SpectrumDataset` with a known injected proton `PowerLawParticleDistribution` + `n_H`,
fit it back with `AafragGammaSpectralModel` via `gammapy.modeling.Fit`, and confirm the
injected parameters are recovered within their fitted statistical uncertainty.

This goes beyond `test_fit_smoke` (Step 5), which only checks that `Fit.run` completes
without raising -- this notebook checks the fit actually converges to the *right answer*.

`target_composition`/`n_H` are not fittable in the same call as the primary spectrum's own
amplitude (ADR-008 notes this is a real degeneracy: flux scales as
`amplitude * n_H / distance**2`, so fitting `amplitude` and `n_H` simultaneously with no
other constraint is unidentifiable). `n_H` and `distance` are therefore frozen at their
(known) true values here, matching the standard real-world use case where the target
density comes from an independent measurement (e.g. HI/CO survey column density) and only
the cosmic-ray spectrum itself (`amplitude`, `index`) is fit to gamma-ray data.

In [ ]:
import numpy as np
from astropy import units as u

from gammapy.datasets import SpectrumDataset
from gammapy.maps import MapAxis, RegionGeom
from gammapy.modeling import Fit
from gammapy.modeling.models import SkyModel

from aafrag_gammapy.models import AafragGammaSpectralModel, PowerLawParticleDistribution

import matplotlib.pyplot as plt
%matplotlib inline

## Simulate a dataset with a known injected spectrum

In [ ]:
energy_axis = MapAxis.from_energy_bounds("300 GeV", "300 TeV", nbin=15, name="energy")
energy_axis_true = MapAxis.from_energy_bounds(
    "100 GeV", "1000 TeV", nbin=25, name="energy_true"
)
geom = RegionGeom.create("icrs;circle(0,0,0.1)", axes=[energy_axis])

dataset = SpectrumDataset.create(geom=geom, energy_axis_true=energy_axis_true)
dataset.exposure.data += 3e9  # cm2 s, chosen for a few 1e4 counts (stable fit, realistic noise)
dataset.mask_safe.data[...] = True  # SpectrumDataset.create()'s mask_safe defaults to all-False

amplitude_true = 1e46 * u.Unit("1/TeV")
index_true = 2.3
n_H_true = 1e3 * u.cm**-3
distance_true = 0.1 * u.kpc

primary_true = PowerLawParticleDistribution(amplitude=amplitude_true, index=index_true)
spectral_true = AafragGammaSpectralModel(primary_true, n_H=n_H_true, distance=distance_true)
sky_true = SkyModel(spectral_model=spectral_true, name="source")

dataset.models = [sky_true]
dataset.fake(random_state=0)

print(f"total simulated counts: {dataset.counts.data.sum():.0f}")

## Fit back with a perturbed starting point

`n_H` and `distance` frozen at their true values (see markdown above); `amplitude`/`index`
start 50%/0.3 away from truth.

In [ ]:
primary_fit = PowerLawParticleDistribution(
    amplitude=1.5 * amplitude_true, index=index_true + 0.3
)
spectral_fit = AafragGammaSpectralModel(primary_fit, n_H=n_H_true, distance=distance_true)
spectral_fit.n_H.frozen = True
sky_fit = SkyModel(spectral_model=spectral_fit, name="source")
dataset.models = [sky_fit]

fit = Fit()
result = fit.run([dataset])
print(result)

## Parameter recovery

In [ ]:
amplitude_fit = spectral_fit.parameters["amplitude"]
index_fit = spectral_fit.parameters["index"]

amplitude_pull = (amplitude_fit.quantity - amplitude_true) / (amplitude_fit.error * amplitude_fit.unit)
index_pull = (index_fit.quantity - index_true) / index_fit.error

print(f"amplitude: fit={amplitude_fit.quantity:.4g} +/- {amplitude_fit.error:.2g} {amplitude_fit.unit}"
      f"  true={amplitude_true:.4g}  pull={amplitude_pull:.2f} sigma")
print(f"index:     fit={index_fit.quantity:.4f} +/- {index_fit.error:.4f}"
      f"  true={index_true}  pull={index_pull:.2f} sigma")

assert result.success
assert abs(amplitude_pull) < 5, "amplitude not recovered within 5 sigma"
assert abs(index_pull) < 5, "index not recovered within 5 sigma"


## Visual check: data vs. best-fit model

In [ ]:
ax = dataset.plot_fit()
plt.show()

## Conclusion

Both free parameters (`amplitude`, `index`) are recovered within a few sigma of their
statistical uncertainty from a starting point deliberately offset by 50%/+0.3, and
`dataset.plot_fit()` shows the best-fit model tracking the simulated counts across the full
energy range with no systematic residual trend. This is the closed-loop confirmation that
`kernel.py`'s cross-section machinery, `models.py`'s `evaluate()`/parameter-aggregation, and
gammapy's own `Fit`/`Dataset` machinery compose correctly end-to-end -- not just that
`Fit.run()` avoids raising (`test_fit_smoke`, Step 5), but that it converges to the actual
injected physics.